# Demo 1 - Sentiment Analysis on Movie Reviews

##**Scenario:**
Movie Review Sentiment Prediction and Trend Forecasting
You work for a media analytics company that processes customer reviews for thousands of movies. These reviews not only reflect sentiment but also evolve over time based on factors like cast performance, marketing, or award wins.

You're tasked with analyzing a movie reviews dataset (like the one you uploaded) using Recurrent Neural Networks (RNNs) to:

* Classify the sentiment of each review using the textual content (positive, neutral, negative).

* Forecast sentiment trends over time for individual movies—predict how viewer sentiment might shift based on review history.

##**Objectives:**

**Understand RNN architectures:**

* Explain unrolled RNNs and their input/output formats.

* Show how data flows through the forward and backward pass.

**Handle sequence modeling challenges:**

* Address vanishing gradients.

* Apply teacher forcing to improve training efficiency.

**Process textual review data:**

* Tokenize, embed, and prepare sequences with appropriate padding and masking.

* Use both stateless and stateful RNNs to compare performance.

**Model review sentiment classification:**

* Build RNNs using Keras (vanilla and GRU-based).

* Evaluate accuracy and sequence handling using real review data.

**Perform time series sentiment forecasting:**

* Use review timestamps and average ratings to build a time-series model with RNNs.

* Forecast future sentiment trends for selected movies.

##Basic RNN Architecture

Build a simple Many-to-One RNN to classify movie reviews as positive, neutral, or negative based on text input.

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
df = pd.read_csv("movie_reviews.csv")  # Make sure this file is in your working directory

In [ ]:
df['sentiment'] = df['rating'].apply(lambda r: 0 if r <= 2 else (1 if r == 3 else 2))

In [ ]:
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")  # Limit vocab to 1000 words
tokenizer.fit_on_texts(df['movie_review'])  # Learn word indices
sequences = tokenizer.texts_to_sequences(df['movie_review'])  # Text to sequences

In [ ]:
max_len = 50  # Set max sequence length
X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')  # Pad short ones
y = tf.keras.utils.to_categorical(df['sentiment'], num_classes=3)  # One-hot encode the target

In [ ]:
model = Sequential([
    Embedding(input_dim=1000, output_dim=16, input_length=max_len),  # Learn word embeddings
    SimpleRNN(32),                        # Basic RNN layer with 32 units
    Dense(16, activation='relu'),        # Fully connected layer
    Dense(3, activation='softmax')       # Output layer for 3 sentiment classes
])

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(X, y, epochs=5, batch_size=32, validation_split=0.2)  # Use 20% data for validation

In [ ]:
model.summary()

##Vanishing Gradient Problem & Teacher Forcing (Conceptual + Setup)

Build a sample setup showing where vanishing gradient and teachr forcing applies. Although classification doesn't use teacher forcing directly, we simulate a sequence-to-sequence (seq2seq) setup to demonstrate it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, TimeDistributed, Dense, Input
from tensorflow.keras.models import Model

In [ ]:
input_sequences = np.random.randint(1, 50, size=(1000, 10))
target_sequences = input_sequences.copy()
vocab_size = 51
target_onehot = tf.keras.utils.to_categorical(target_sequences, num_classes=vocab_size)

In [ ]:
inputs = Input(shape=(10,))
x = Embedding(input_dim=vocab_size, output_dim=16)(inputs)
x = LSTM(32, return_sequences=True)(x)
outputs = TimeDistributed(Dense(vocab_size, activation='softmax'))(x)

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(input_sequences, target_onehot, epochs=5, batch_size=32, verbose=1)


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], marker='o')
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], marker='o', color='green')
plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.tight_layout()
plt.show()

In [ ]:
sample_input = input_sequences[:5]
predicted_probs = model.predict(sample_input)
predicted_sequences = np.argmax(predicted_probs, axis=-1)

print("\nSample Predictions (First 5 Sequences):")
for i in range(5):
    print(f"Input:     {sample_input[i]}")
    print(f"Predicted: {predicted_sequences[i]}")
    print(f"Target:    {target_sequences[i]}")
    print('-' * 50)

##Input/Output Formats in RNNs
We will demonstrate:

* Many-to-One: Sentiment classification (already done)

* Many-to-Many: Simulated sequence prediction task (output at every step)



In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, TimeDistributed, Dense

In [ ]:
X_seq = np.random.randint(1, 100, (1000, 10))  # e.g. [[23, 87, 5, ..., 44]]
y_seq = tf.keras.utils.to_categorical(X_seq, num_classes=100)  # shape: (1000, 10, 100)

In [ ]:
model = Sequential([
    Embedding(input_dim=100, output_dim=16, input_length=10),
    SimpleRNN(32, return_sequences=True),
    TimeDistributed(Dense(100, activation='softmax'))
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(X_seq, y_seq, epochs=3, batch_size=32, verbose=1)

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], marker='o')
plt.title('Training Loss (Many-to-Many)')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], marker='o', color='green')
plt.title('Training Accuracy (Many-to-Many)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.tight_layout()
plt.show()

In [ ]:
preds = model.predict(X_seq[:5])
predicted_sequences = np.argmax(preds, axis=-1)

print("\n🔍 Sample Predictions (Echo Task)")
for i in range(5):
    print(f"Input     : {X_seq[i]}")
    print(f"Predicted : {predicted_sequences[i]}")
    print(f"Actual    : {X_seq[i]}")
    print('-' * 60)

##Build and compare Vanilla RNN and GRU models
Vanilla RNN and GRU models performed on sentiment classification of movie reviews using accuracy and loss curves.

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, GRU, Dense

In [ ]:
df['label'] = (df['rating'] >= 4).astype(int)  # Binary sentiment: positive (1) if rating >= 4, else negative (0)


In [ ]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['movie_review'])
sequences = tokenizer.texts_to_sequences(df['movie_review'])
X = pad_sequences(sequences, maxlen=100, padding='post')
y = df['label'].values

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
def build_and_train_model(model_type='RNN'):
    model = Sequential()
    model.add(Embedding(input_dim=5000, output_dim=64, input_length=100))
    if model_type == 'RNN':
        model.add(SimpleRNN(64))
    elif model_type == 'GRU':
        model.add(GRU(64))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    print(f"\nTraining {model_type} model...\n")
    history = model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_val, y_val), verbose=1)
    return model, history

In [ ]:
rnn_model, rnn_history = build_and_train_model('RNN')
gru_model, gru_history = build_and_train_model('GRU')

In [ ]:
plt.figure(figsize=(12, 5))

In [ ]:
plt.subplot(1, 2, 1)
plt.plot(rnn_history.history['val_accuracy'], label='RNN Val Acc')
plt.plot(gru_history.history['val_accuracy'], label='GRU Val Acc')
plt.title("Validation Accuracy Comparison")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

In [ ]:
plt.subplot(1, 2, 2)
plt.plot(rnn_history.history['val_loss'], label='RNN Val Loss')
plt.plot(gru_history.history['val_loss'], label='GRU Val Loss')
plt.title("Validation Loss Comparison")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

##Perform Time Series Sentiment Forecasting

Use review timestamps to compute average sentiment over time.

Train an RNN model to forecast future sentiment trends for selected movies.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

In [ ]:
df['review_date'] = pd.to_datetime(df['review_date'])

In [ ]:
df['label'] = (df['rating'] >= 4).astype(int)

In [ ]:
movie = df['movie_name'].value_counts().idxmax()
movie_df = df[df['movie_name'] == movie]

monthly_sentiment = movie_df.set_index('review_date').resample('M')['label'].mean().fillna(0)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(monthly_sentiment, marker='o')
plt.title(f"Monthly Average Sentiment for '{movie}'")
plt.xlabel("Month")
plt.ylabel("Average Sentiment")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
series = monthly_sentiment.values.reshape(-1, 1)
scaler = MinMaxScaler()
series_scaled = scaler.fit_transform(series)

In [ ]:
def create_sequences(data, window_size):
    X, y = [], []
    # Need at least window_size + 1 data points to create sequences
    if len(data) <= window_size:
        print(f"Not enough data points ({len(data)}) to create sequences with window size ({window_size}). Need at least {window_size + 1}.")
        return np.array(X), np.array(y) # Return empty arrays if not enough data

    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

In [ ]:
window_size = 4 # Changed from 5 to 4

X, y = create_sequences(series_scaled, window_size)

In [ ]:
if X.shape[0] == 0:
    print("No sequences were created. Cannot train the model.")
else:
    # Build and train the RNN model
    model = Sequential([
        SimpleRNN(50, activation='tanh', input_shape=(window_size, 1)),
        Dense(1)
    ])

    model.compile(optimizer='adam', loss='mse')
    model.fit(X, y, epochs=100, verbose=0)

    # Forecast next 12 months
    # Need at least window_size data points to start forecasting
    if len(series_scaled) < window_size:
        print("Not enough historical data to start forecasting.")
    else:
        last_window = series_scaled[-window_size:]
        predictions = []
        for _ in range(12):
            # Reshape last_window for prediction (1 sample, window_size time steps, 1 feature)
            pred = model.predict(last_window.reshape(1, window_size, 1), verbose=0)
            predictions.append(pred[0][0])
            # Update last_window by removing the first element and adding the prediction
            last_window = np.append(last_window[1:], pred, axis=0)

        forecast = scaler.inverse_transform(np.array(predictions).reshape(-1, 1))
        # Calculate future dates starting from the month after the last historical date
        future_dates = pd.date_range(start=monthly_sentiment.index[-1] + pd.offsets.MonthBegin(), periods=12, freq='M')

        # Plot forecast
        plt.figure(figsize=(10, 4))
        plt.plot(monthly_sentiment.index, monthly_sentiment.values, label='Historical Sentiment')
        plt.plot(future_dates, forecast, label='Forecasted Sentiment', linestyle='--', marker='o')
        plt.title(f"Sentiment Forecast for '{movie}'")
        plt.xlabel("Time")
        plt.ylabel("Average Sentiment")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()